# 07 — Explicabilité du LightGBM final

> **Rappel : SHAP explique le modèle, pas la causalité.** Une contribution positive signifie « **selon le modèle**, cette caractéristique est associée à un risque plus élevé ». Elle ne dit pas que la modifier ferait baisser le risque : ce sont des associations apprises sur des données d'observation.

**Modèle expliqué** : le LightGBM final (E9, `models/pipeline.joblib`), base du modèle calibré (E10). SHAP travaille sur l'échelle **log-odds** du modèle brut ; la calibration sigmoïde étant monotone, elle ne change ni le classement ni le sens des effets.

**Données** : clients du **train** uniquement (le test n'est pas relu).

**Méthodes** :
- **importance par permutation** : baisse d'AUC quand on mélange une variable d'origine, mesurée sur un fold de validation (modèle entraîné sur les 4 autres folds) ;
- **SHAP** (`TreeExplainer`, valeurs exactes) : contribution de chaque variable au score de chaque client ; les features dérivées de E6 sont **regroupées sur leur variable d'origine** (somme des contributions, exacte car SHAP est additif).

In [1]:
import joblib
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display
from sklearn.inspection import permutation_importance

from churn.charts import beeswarm_chart, dependence_grid, horizontal_bar_chart, publish, waterfall_chart
from churn.config import get_config
from churn.data.split import load_train
from churn.evaluation.cv import make_folds
from churn.explain.reason_codes import reason_codes, reasons_table
from churn.explain.shap_utils import LABELS, ShapExplainer, global_importance, label, source_variable
from churn.logging_setup import setup_logging
from churn.models.factory import build_pipeline

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

cfg = get_config()
train = load_train()
y = train[cfg.data.target]
folds = make_folds(y)
pipeline = joblib.load(cfg.paths.models_dir / "pipeline.joblib")
final_model = joblib.load(cfg.paths.models_dir / "final_model.joblib")
print(f"modèle : {cfg.models.final} (calibration {final_model.method}), {len(train)} clients du train")

C:\Users\malek\OneDrive\Desktop\telecom_churn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modèle : lightgbm (calibration sigmoid), 80000 clients du train


## 1. Importance par permutation (fold de validation)

Modèle LightGBM final entraîné sur 4 folds, évalué sur le 5e (16 000 clients jamais vus). Pour chaque **variable d'origine**, on mélange ses valeurs et on mesure la baisse d'AUC (5 répétitions). Mélanger une variable d'origine perturbe aussi toutes les features qui en dérivent.

In [2]:
train_idx, valid_idx = folds[0]
fold_model = build_pipeline(cfg.models.final).fit(train.iloc[train_idx], y.iloc[train_idx])
X_valid = train.iloc[valid_idx].drop(columns=[cfg.data.target])
perm = permutation_importance(fold_model, X_valid, y.iloc[valid_idx], scoring="roc_auc",
                              n_repeats=5, random_state=cfg.random_state, n_jobs=1)
perm_df = pd.DataFrame({"baisse_auc": perm.importances_mean, "ecart_type": perm.importances_std},
                       index=X_valid.columns).sort_values("baisse_auc", ascending=False)
perm_df["libelle"] = [label(c) for c in perm_df.index]
display(perm_df.head(15).round(5))
top_perm = perm_df.head(15)
publish(horizontal_bar_chart(1000 * top_perm.set_index("libelle")["baisse_auc"],
                             "Mélanger l'âge du terminal ou l'ancienneté fait le plus chuter l'AUC",
                             "baisse d'AUC (millièmes)", error=1000 * top_perm.set_index("libelle")["ecart_type"], fmt=".1f"),
        "07_permutation", width=1000)

,baisse_auc,ecart_type,libelle
months,0.03773,0.00158,Ancienneté
eqpdays,0.03445,0.00135,Âge du terminal
change_mou,0.02895,0.00204,Évolution de l'usage
avg3mou,0.01771,0.00071,Usage récent (3 mois / 6 mois)
mou_Mean,0.01347,0.00041,Minutes d'appel par mois
totmrc_Mean,0.01171,0.00087,Forfait mensuel
avg6mou,0.00883,0.00087,Minutes moyennes sur 6 mois
crclscod,0.00866,0.00079,Classe de crédit
rev_Mean,0.00839,0.00036,Facture mensuelle
uniqsubs,0.00723,0.00061,Lignes ouvertes


**Lecture** : mélanger l'**ancienneté** fait perdre 37,7 millièmes d'AUC, l'**âge du terminal** 34,5, l'**évolution de l'usage** 29,0 ; viennent ensuite l'usage récent (17,7), les minutes d'appel (13,5) et le forfait (11,7). Au-delà du 10e rang, chaque variable pèse moins de 7 millièmes. La permutation mesure ce que le modèle **perd** sans une variable ; les variables redondantes se partagent l'importance (mélanger l'une laisse l'information dans l'autre), ce qui sous-estime par exemple `hnd_price`, corrélé à `eqpdays` et `phones`.

## 2. SHAP : importance globale

10 000 clients du train tirés au hasard. Importance = moyenne de |SHAP| (log-odds) par variable d'origine.

In [3]:
sample = train.sample(10_000, random_state=cfg.random_state)
explainer = ShapExplainer(pipeline)
result = explainer.explain(sample)
check = (result.log_odds() - pipeline.named_steps["model"].predict(result.features, raw_score=True)).abs().max()
print(f"valeur de base (log-odds moyen) : {result.base_value:+.3f} ; reconstitution exacte : écart max {check:.1e}")
imp = global_importance(result)
display(imp.head(15).round(4))
publish(horizontal_bar_chart(1000 * imp.head(15).set_index("libelle")["shap_abs_moyen"],
                             "L'âge du terminal, l'évolution de l'usage et l'ancienneté dominent les scores",
                             "|SHAP| moyen (millièmes de log-odds)", fmt=".0f"), "07_shap_importance", width=1000)

valeur de base (log-odds moyen) : -0.022 ; reconstitution exacte : écart max 5.8e-15


,shap_abs_moyen,libelle,part_pct
eqpdays,0.2611,Âge du terminal,12.2
change_mou,0.1584,Évolution de l'usage,7.4
months,0.1562,Ancienneté,7.3
mou_Mean,0.1011,Minutes d'appel par mois,4.7
avg3mou,0.0964,Usage récent (3 mois / 6 mois),4.5
crclscod,0.0924,Classe de crédit,4.3
totmrc_Mean,0.0883,Forfait mensuel,4.1
hnd_price,0.0807,Prix du terminal,3.8
lor,0.0737,Durée de résidence,3.4
area,0.0695,Région,3.2


### Beeswarm (features du modèle)

Un point par client : sa contribution SHAP (axe horizontal) et le rang de la valeur de la variable (couleur : bleu = valeur basse, rouge = valeur haute ; gris = valeur manquante ou catégorielle).

In [4]:
feat_imp = global_importance(result, grouped=False)
top_features = feat_imp.head(15).index.tolist()
feature_labels = {c: (label(c) if c == source_variable(c) else f"{label(source_variable(c))} ({c})")
                  for c in result.values.columns}
swarm = result.values.index[:3000]   # 3 000 clients suffisent pour lire les formes (notebook léger)
publish(beeswarm_chart(result.values.loc[swarm], result.features.loc[swarm], top_features, feature_labels,
                       "Terminal ancien, usage en baisse et forfait modeste poussent le score vers le haut"),
        "07_beeswarm", width=1100)

**Lecture** : l'**âge du terminal** domine (12,2 % de l'importance SHAP totale), devant l'**évolution de l'usage** (7,4 %) et l'**ancienneté** (7,3 %) ; puis les minutes d'appel, l'usage récent, la classe de crédit, le forfait et le prix du terminal (4 à 5 % chacun). Aucune variable ne pèse plus de 13 % : comme prévu en E5, le score **combine de nombreux petits effets**.

Le beeswarm donne le sens des effets : terminal ancien (rouge) à droite, usage élevé (rouge) à gauche, forfait élevé à gauche. Pour `change_mou`, les valeurs basses (usage en baisse, bleu) poussent vers le haut. Ces sens concordent avec l'EDA.

## 3. Dependence plots des 5 premières variables

Contribution SHAP en fonction de la valeur de la variable (3 000 clients). Pour `months` et `eqpdays`, on vérifie en plus l'effet **regroupé** (variable d'origine + features dérivées : `in_contract_end`, `handset_old`, `eqpdays_per_tenure_day`).

In [5]:
top5 = feat_imp.head(5).index.tolist()
sub = result.values.index[:3000]
publish(dependence_grid(result.values.loc[sub], result.features.loc[sub], top5, feature_labels,
                        "Effets non linéaires : seuils et plateaux plutôt que pentes régulières"),
        "07_dependence_top5", width=1100)
grouped = result.grouped()
months_effect = grouped["months"].groupby(result.features["months"].clip(upper=30)).mean()
eqp_bins = pd.cut(result.features["eqpdays"], [0, 180, 270, 290, 300, 310, 330, 400, 600, 900, 2000], right=False)
eqp_effect = grouped["eqpdays"].groupby(eqp_bins, observed=True).agg(["mean", "size"])
display(pd.DataFrame({"effet SHAP moyen (ancienneté, regroupé)": months_effect.round(3)}).T)
display(eqp_effect.rename(columns={"mean": "effet SHAP moyen (terminal, regroupé)", "size": "clients"}).round(3))
publish(dependence_grid(pd.DataFrame({"months": grouped["months"], "eqpdays": grouped["eqpdays"]}).loc[sub],
                        result.features.loc[sub, ["months", "eqpdays"]], ["months", "eqpdays"],
                        {"months": "Ancienneté (effet regroupé)", "eqpdays": "Âge du terminal (effet regroupé)"},
                        "Le pic à 11-12 mois et la marche à 300 jours sont retrouvés par le modèle", cols=2),
        "07_dependence_months_eqpdays", width=1100)

months,6,7,8,9,10,11,12,13,14,15,...,21,22,23,24,25,26,27,28,29,30
"effet SHAP moyen (ancienneté, regroupé)",-0.328,-0.313,-0.3,-0.317,-0.134,0.512,0.486,0.118,0.082,0.064,...,-0.025,-0.025,-0.029,-0.039,-0.042,-0.043,-0.045,-0.045,-0.051,-0.08


,"effet SHAP moyen (terminal, regroupé)",clients
eqpdays,,
"[0, 180)",-0.368,1858
"[180, 270)",-0.326,1723
"[270, 290)",-0.314,303
"[290, 300)",-0.309,173
"[300, 310)",0.047,211
"[310, 330)",0.215,489
"[330, 400)",0.188,1318
"[400, 600)",0.195,1988
"[600, 900)",0.206,1453


**Lecture** :

- **Ancienneté (effet regroupé)** : autour de −0,31 de 6 à 9 mois, −0,13 à 10 mois, **pic à +0,51 à 11 mois et +0,49 à 12 mois**, puis retour rapide vers 0 et légère décroissance (−0,08 à 30 mois). C'est exactement la forme de l'EDA (E4) : churn faible sous engagement, pic à l'échéance.
- **Âge du terminal (effet regroupé)** : plat vers −0,31 jusqu'à 299 jours, puis **marche** : +0,05 de 300 à 309 jours, **+0,22 de 310 à 330 jours**, et un plateau autour de +0,20, qui remonte à +0,26 au-delà de 900 jours. Le seuil de 300 jours de l'EDA est retrouvé ; le modèle le place même un peu après (310 jours), là où le taux de churn saute en EDA (305-309 jours).
- Les autres dependence plots montrent surtout des **seuils et plateaux** : c'est ce que les arbres représentent naturellement, et ce qu'une régression logistique ne capte qu'avec des features dédiées (E6, E7).

In [6]:
# Deux vérifications : d'où vient l'importance de change_mou ? dans quel sens joue lor ?
cm = result.values["change_mou"]
is_nan = result.features["change_mou"].isna()
print(f"change_mou : {cm[is_nan].abs().sum() / cm.abs().sum():.1%} de son |SHAP| vient des NaN "
      f"({is_nan.mean():.1%} des clients ; SHAP moyen des NaN {cm[is_nan].mean():+.3f})")
display(cm.groupby(pd.qcut(result.features["change_mou"], 10), observed=True).mean().round(3)
          .rename("SHAP moyen par décile de change_mou").to_frame().T)
display(result.values["lor"].groupby(result.features["lor"]).mean().round(3)
          .rename("SHAP moyen de lor (années de résidence)").to_frame().T)

change_mou : 3.6% de son |SHAP| vient des NaN (0.8% des clients ; SHAP moyen des NaN +0.675)


change_mou,"(-2867.501, -234.625]","(-234.625, -119.25]","(-119.25, -63.0]","(-63.0, -29.0]","(-29.0, -6.5]","(-6.5, 9.25]","(9.25, 39.75]","(39.75, 92.5]","(92.5, 197.375]","(197.375, 2067.5]"
SHAP moyen par décile de change_mou,0.441,0.198,0.044,-0.06,-0.097,-0.142,-0.174,-0.158,-0.138,-0.067


lor,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0
SHAP moyen de lor (années de résidence),0.089,0.081,0.081,0.014,-0.012,-0.041,-0.062,-0.08,-0.08,-0.081,-0.099,-0.104,-0.112,-0.131,-0.143,-0.154


**Deux vérifications** :

- **`change_mou`** (2e en SHAP, seulement 9e en E5) : **3,6 %** seulement de son importance vient des valeurs manquantes (0,8 % des clients, le signal « déjà parti » de D32). L'essentiel vient des **vraies valeurs** : une forte baisse d'usage (plus de 235 minutes perdues) pèse **+0,44**, une hausse −0,17. C'est donc bien un signal de **baisse d'usage**, pas une fuite. Pourquoi plus fort qu'en E5 ? Seule, la variation d'usage dit peu de chose ; rapportée au niveau d'usage (que le modèle connaît), une même baisse devient plus parlante : c'est un effet d'**interaction**.
- **`lor`** (durée de résidence, 9e en SHAP, 44e en E5 et **non significative** à elle seule après Holm en E2) : effet monotone, de +0,09 pour 0 an à −0,15 pour 15 ans. À autres variables égales, une résidence ancienne va avec moins de churn : probable indicateur de **stabilité** du foyer. C'est une association conditionnelle, pas un levier.

## 4. Trois clients expliqués

Clients du train choisis selon leur probabilité calibrée (modèle final, E10) : **High** (97,5e centile), **Medium** (médiane), **Low** (2,5e centile). Les niveaux définitifs seront fixés en E12 selon la capacité de campagne.

Pour chacun : probabilité calibrée sur l'échantillon, probabilité ramenée au taux réel **supposé** (2 % par mois, hypothèse), cascade SHAP et **3 codes de raisons**.

In [7]:
p_sample = pd.Series(final_model.proba_sample(sample), index=sample.index)
p_real = pd.Series(final_model.proba_real(sample), index=sample.index)
targets = {"High": 0.975, "Medium": 0.50, "Low": 0.025}
clients = {name: (p_sample - p_sample.quantile(q)).abs().idxmin() for name, q in targets.items()}
summary = pd.DataFrame({name: {"Customer_ID": int(sample.loc[i, cfg.data.id_col]),
                               "probabilité (échantillon)": round(p_sample[i], 3),
                               "probabilité au taux réel (hypothèse 2 %)": round(p_real[i], 4),
                               "centile de risque": round(100 * (p_sample < p_sample[i]).mean(), 1),
                               "churn observé": int(sample.loc[i, cfg.data.target])}
                        for name, i in clients.items()}).T
summary

,Customer_ID,probabilité (échantillon),probabilité au taux réel (hypothèse 2 %),centile de risque,churn observé
High,1062189.0,0.818,0.0855,97.5,1.0
Medium,1001503.0,0.497,0.0201,50.0,0.0
Low,1084783.0,0.164,0.0040,2.5,1.0


In [8]:
for name, idx in clients.items():
    contrib = result.grouped().loc[idx].rename(index=label)
    title = (f"Client {name} (id {int(sample.loc[idx, cfg.data.id_col])}) : "
             f"{100 * p_real[idx]:.1f} % de risque mensuel au taux réel supposé de 2 %".replace(".", ","))
    publish(waterfall_chart(contrib, result.base_value, title), f"07_waterfall_{name.lower()}", width=1000)
    print(f"--- Client {name}")
    display(reasons_table(reason_codes(result, idx, top=3)))

--- Client High


,facteur,situation du client,effet,contribution (log-odds)
0,Minutes d'appel par mois,Minutes d'appel : 0 par mois,augmente le risque,0.678
1,Usage récent (3 mois / 6 mois),Usage en baisse de 45 % (3 derniers mois contre 6 mois),augmente le risque,0.342
2,Classe de crédit,Classe de crédit EA,réduit le risque,-0.316


--- Client Medium


,facteur,situation du client,effet,contribution (log-odds)
0,Âge du terminal,Terminal ancien : 43 mois,augmente le risque,0.261
1,Appels par mois depuis l'ouverture,"Appels par mois depuis l'ouverture : 11,2",réduit le risque,-0.235
2,Minutes d'appel par mois,Minutes d'appel : 9 par mois,augmente le risque,0.198


--- Client Low


,facteur,situation du client,effet,contribution (log-odds)
0,Ancienneté,Ancienneté : 8 mois,réduit le risque,-0.408
1,Âge du terminal,Terminal récent : 8 mois,réduit le risque,-0.380
2,Prix du terminal,Terminal à 200 $,réduit le risque,-0.152


**Lecture des trois clients** :

- **High** (97,5e centile, 0,82 sur l'échantillon, **8,6 % par mois au taux réel supposé**) : premier facteur, **0 minute d'appel par mois** (+0,68), puis un usage en baisse de 45 % sur 3 mois. C'est le profil d'un client qui **n'utilise presque plus** son abonnement. Comme en E8 (`mou_Mean` = 0 : 79 % de churn), ce client est peut-être déjà en train de partir : une campagne devra se demander s'il est encore récupérable. Ce client a effectivement churné.
- **Medium** (médiane, 2,0 % par mois) : terminal ancien de 43 mois (+0,26) et très faible usage (+0,20), compensés par un historique d'appels élevé (−0,24). Des facteurs de sens opposés s'annulent : c'est typique d'un score moyen. Il n'a pas churné.
- **Low** (2,5e centile, 0,4 % par mois) : **8 mois d'ancienneté** (sous engagement, −0,41), **terminal récent de 8 mois** (−0,38), terminal haut de gamme à 200 $ (−0,15). Il a pourtant **churné** : une probabilité faible n'est pas une certitude ; sur des milliers de clients « Low », quelques-uns partent.

Les codes de raisons sont directement lisibles par un conseiller ; ils décrivent la situation du client **selon le modèle**, pas la cause de son départ.

## 5. Confrontation avec l'EDA (E4) et les tests statistiques (E5)

In [9]:
stats = pd.read_csv(cfg.paths.reports_dir / "stats_univariate.csv", index_col=0)
perm_rank = perm_df["baisse_auc"].rank(ascending=False)
rows = []
for rank, (var, r) in enumerate(imp.head(15).iterrows(), start=1):
    rows.append({"variable": var, "libellé": r["libelle"], "rang SHAP": rank,
                 "rang permutation": int(perm_rank.get(var, np.nan)) if var in perm_rank else None,
                 "rang E5 (info. mutuelle)": int(stats.loc[var, "rang"]) if var in stats.index else None,
                 "effet E5 (r ou V)": round(stats.loc[var, "effet"], 3) if var in stats.index else None,
                 "significatif E5 (Holm)": bool(stats.loc[var, "significatif_holm"]) if var in stats.index else None})
confront = pd.DataFrame(rows).set_index("variable")
confront

,libellé,rang SHAP,rang permutation,rang E5 (info. mutuelle),effet E5 (r ou V),significatif E5 (Holm)
variable,,,,,,
eqpdays,Âge du terminal,1,2,2,0.149,True
change_mou,Évolution de l'usage,2,3,9,-0.061,True
months,Ancienneté,3,1,1,0.053,True
mou_Mean,Minutes d'appel par mois,4,5,15,-0.079,True
avg3mou,Usage récent (3 mois / 6 mois),5,4,13,-0.067,True
crclscod,Classe de crédit,6,8,23,0.069,True
totmrc_Mean,Forfait mensuel,7,6,3,-0.084,True
hnd_price,Prix du terminal,8,12,4,-0.118,True
lor,Durée de résidence,9,15,44,-0.052,True


**Cohérences avec l'EDA et E5** :

- Le **trio de tête** est le même selon les trois méthodes (SHAP, permutation, information mutuelle en E5) : **âge du terminal, ancienneté, évolution de l'usage**. Le pic à 11-12 mois et la marche à 300 jours de l'EDA sont retrouvés par le modèle.
- Les **sens** des effets concordent avec l'EDA : terminal ancien et bon marché, forfait modeste, usage faible ou en baisse → plus de risque ; classe de crédit E (−0,33) et région DC / Maryland / Virginia → moins de risque.

**Surprises (effets conditionnels)** : des variables faibles seules deviennent utiles à autres variables égales.

- `drop_vce_Mean` (appels coupés) : 103e en E5, **11e** en SHAP. Seuls, les appels coupés vont avec les gros utilisateurs, qui churnent moins ; à usage égal, ils poussent au churn (comme dans la LR en E7).
- `crclscod` (23e → 6e), `area` (39e → 10e), `lor` (44e → 9e) : effets de catégories ou de stabilité faibles seuls, mais qui se combinent aux autres.
- `avgqty` (non significatif en E5, 12e en SHAP) : à usage récent égal, un historique d'appels élevé signale une **baisse** d'usage (déjà vu en E7 dans les coefficients de la LR).
- `hnd_price` (4e en E5, 8e en SHAP, 12e en permutation) : son information est en partie portée par `eqpdays` et `phones`, avec lesquels il est corrélé.

La leçon rejoint E5 : **l'importance univariée et l'importance dans le modèle ne se confondent pas** ; un modèle multivarié exploite des effets conditionnels et des interactions.

## Conclusion

1. **Trois facteurs dominent**, quelle que soit la méthode : **âge du terminal** (12 % de l'importance SHAP), **évolution de l'usage** (7 %), **ancienneté** (7 %) ; le reste est une somme de petits effets.
2. Le modèle retrouve les **formes de l'EDA** : pic de risque à 11-12 mois d'ancienneté, marche vers 300-310 jours d'âge du terminal, risque croissant avec la baisse d'usage.
3. `change_mou` est un vrai signal de baisse d'usage (seulement 3,6 % de son importance vient des NaN « déjà parti »).
4. Des effets **conditionnels** apparaissent (appels coupés, classe de crédit, région, durée de résidence) : invisibles ou faibles seuls, utiles à autres variables égales.
5. **Codes de raisons** : 3 facteurs par client en phrases métier, avec leur sens ; ils serviront à la fiche client (API, dashboard) et à l'assistant.
6. **Rappel** : SHAP explique le **modèle** ; ces associations ne sont pas des causes, et agir sur un facteur ne garantit pas de réduire le risque. Le client High illustre aussi une limite métier : un client à usage nul est « détecté », mais peut-être déjà perdu.